**Figure 4: Tail Latency Scaling with Trajectory Length.** Per-call and per-trajectory
p50/p95 at 54/64/96 calls for the two AgentTX modes. Results suggest that median
per-call latency stays nearly flat as trajectories grow, while the p95 gap between
full and no-trace holds at ~2.2-2.3x -- the read-trace tax neither amortizes nor
compounds with length, pinning it as a per-call, capture-side cost.


In [ ]:
# ipython -c "%run plot_tail_scaling.ipynb"
# FAST/USENIX line-plot conventions: white panels, boxed top legend, red solid squares = ours.
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, cm

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

OURS  = dict(color='#c00000', marker='s', linestyle='-',  linewidth=1.0, markersize=3.2)
BASE1 = dict(color='#e78129', marker='x', linestyle=':',  linewidth=0.9, markersize=3.6, markeredgewidth=0.9)
BASE2 = dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none')
REF   = dict(color='black', linestyle='--', linewidth=0.8)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

df = pd.read_csv(RESULTS / 'motivation_tail_scaling.csv')
lengths = sorted(df['length'].unique())

def series(metric, mode):
    return np.asarray([float(df[(df['length'] == l) & (df['mode'] == mode)][metric].iloc[0]) for l in lengths])

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
panels = [
    ('step_p50_ms', 'Per-call p50 (ms)', '(a) Median call latency'),
    ('step_p95_ms', 'Per-call p95 (ms)', '(b) Tail call latency'),
    ('run_p50_ms', 'Trajectory p50 (ms)', '(c) Median trajectory latency'),
    ('run_p95_ms', 'Trajectory p95 (ms)', '(d) Tail trajectory latency'),
]
handles = []
for i, (metric, ylabel, sub) in enumerate(panels):
    ax = plt.subplot(2, 2, i + 1)
    h_nt, = ax.plot(lengths, series(metric, 'agenttx_without_read_tracing'), **BASE2, label='AgentTX no-trace')
    h_fl, = ax.plot(lengths, series(metric, 'agenttx_full'), **OURS, label='AgentTX full (ours)')
    if not handles:
        handles = [h_nt, h_fl]
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xlabel('Trajectory length (# calls)\n' + sub, fontsize=7)
    ax.set_xticks(lengths)
    ax.tick_params(axis='both', labelsize=7)

fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=2,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.4, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.93])
plt.savefig(FIGDIR / 'FIG-Motivation-Tail-Scaling.pdf', bbox_inches='tight', pad_inches=0.02)
plt.savefig(FIGDIR / 'FIG-Motivation-Tail-Scaling.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

p95_gap = series('step_p95_ms', 'agenttx_full') / series('step_p95_ms', 'agenttx_without_read_tracing')
print(f"p95 read-trace gap across lengths: {[f'{v:.2f}x' for v in p95_gap]}")
